# Breast Cancer Subtype Prediction Project
# Multi-Omics Integration (RNA-Seq + Methylation)
# Step 1: Reading & Data Synchronization

## Import Library

In [5]:
import pandas as pd
import numpy as np
import pickle
import os

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 20)

print("✅ Libraries Imported Successfully.")

✅ Libraries Imported Successfully.


## Define Functions

In [ ]:
def save_object(obj, filename):
    """
    Saves an object to the outputs directory.
    Target Directory: ../outputs/
    """
    save_dir = '../outputs'
    
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
        print(f"Directory '{save_dir}' created.")
    
    file_path = os.path.join(save_dir, filename + '.pkl')
    
    with open(file_path, 'wb') as f:
        pickle.dump(obj, f, pickle.HIGHEST_PROTOCOL)
    
    print(f"💾 Saved: {file_path}")

## Load Data & Set Paths

In [ ]:
data_dir = '../data'

clinical_file = os.path.join(data_dir, 'Human_TCGA_BRCA_MS_Clinical_Clinical_01_28_2016_BI_Clinical_Firehose.tsi')
rna_file = os.path.join(data_dir, 'Human_TCGA_BRCA_UNC_RNAseq_HiSeq_RNA_01_28_2016_BI_Gene_Firehose.gz')
meth_file = os.path.join(data_dir, 'Human_TCGA_BRCA_JHU_USC_Methylation_Meth450_01_28_2016_BI_Gene_Firehose.gz')

print(f"Checking for data in: {os.path.abspath(data_dir)}")

if os.path.exists(clinical_file):
    print("✅ Clinical file found.")
else:
    print(f"❌ Error: Clinical file NOT found at: {clinical_file}")

Checking for data in: c:\Users\HYPER_STOCK_TABRIZ\Downloads\BRCA_Subtype_Prediction\data
✅ Clinical file found.


## Process Clinical Data (Extract PAM50 Labels)

In [ ]:
print("Loading Clinical Data...")

try:
    clinical = pd.read_csv(clinical_file, sep='\t', index_col=0).T
except FileNotFoundError:
    raise FileNotFoundError("Run the cell above to check file paths!")

if 'PAM50' not in clinical.columns:
    raise ValueError("Error: Column 'PAM50' Not Found in Clinical Data!")

clinical_clean = clinical.dropna(subset=['PAM50'])

label_mapping = {
    'LumA': 0,
    'LumB': 1,
    'Her2': 2,
    'Basal': 3,
    'Normal': 4
}

clinical_clean = clinical_clean[clinical_clean['PAM50'].isin(label_mapping.keys())]
clinical_clean['label'] = clinical_clean['PAM50'].map(label_mapping).astype(int)

print(f"✅ Clinical Data Processed. Valid Patients: {clinical_clean.shape[0]}")
print("Subtype Distribution:\n", clinical_clean['PAM50'].value_counts())

Loading Clinical Data...
✅ Clinical Data Processed. Valid Patients: 826
Subtype Distribution:
 PAM50
LumA     426
LumB     186
Basal    147
Her2      67
Name: count, dtype: int64


## Load Omics Data (RNA-Seq & Methylation)

In [ ]:
print("Loading Omics Data (This may take a moment)...")

rna = pd.read_csv(rna_file, sep='\t', index_col=0).T
print(f"✅ RNA-Seq Loaded. Shape: {rna.shape}")

meth = pd.read_csv(meth_file, sep='\t', index_col=0).T
print(f"✅ Methylation Loaded. Shape: {meth.shape}")

Loading Omics Data (This may take a moment)...
✅ RNA-Seq Loaded. Shape: (1093, 20155)
✅ Methylation Loaded. Shape: (783, 20106)


## Processing and Matching Samples

In [ ]:
print("Synchronizing patients across all datasets...")

common_patients = clinical_clean.index.intersection(rna.index).intersection(meth.index)

print(f"------------------------------------------------")
print(f"Patients in Clinical: {clinical_clean.shape[0]}")
print(f"Patients in RNA-Seq:  {rna.shape[0]}")
print(f"Patients in Methylation: {meth.shape[0]}")
print(f"------------------------------------------------")
print(f"🚀 FINAL COMMON PATIENTS: {len(common_patients)}")

if len(common_patients) == 0:
    print("❌ Critical Error: No common patients found. Check Patient IDs.")

Synchronizing patients across all datasets...
------------------------------------------------
Patients in Clinical: 826
Patients in RNA-Seq:  1093
Patients in Methylation: 783
------------------------------------------------
🚀 FINAL COMMON PATIENTS: 549


## Saving Result

In [ ]:
if len(common_patients) > 0:
    print("Saving processed datasets to '../outputs/'...")
    
    
    X_rna_final = rna.loc[common_patients].astype(np.float32)
    X_meth_final = meth.loc[common_patients].astype(np.float32)
    y_final = clinical_clean.loc[common_patients, 'label']
    
    
    save_object(X_rna_final, 'X_rna_raw')
    save_object(X_meth_final, 'X_meth_raw')
    save_object(y_final, 'y_labels')
    save_object(common_patients, 'patient_ids')
    
    print("\n🎉 Reading Step Completed Successfully.")
else:
    print("Skipping save due to no common patients.")

Saving processed datasets to '../outputs/'...
💾 Saved: ../outputs\X_rna_raw.pkl
💾 Saved: ../outputs\X_meth_raw.pkl
💾 Saved: ../outputs\y_labels.pkl
💾 Saved: ../outputs\patient_ids.pkl

🎉 Reading Step Completed Successfully.
